# Statistical Significance Layer — Contribution 1 (sealed test, n=109)

Consumes the two per-patient prediction files emitted by the sealed-test notebooks:
`per_patient_sealed_predictions_FS.json` (nine feature-selection/regime configs) and
`per_patient_sealed_predictions_NB3.json` (five architectures). No GPU required.

Tests: McNemar (exact binomial, paired hard labels), DeLong (paired AUROC),
paired bootstrap 95% CI on the MCC delta, Holm correction within each family,
and Fisher's exact single-model discrimination. Family 1 = GraphSAGE vs the four
other architectures; Family 2 = BASELINE vs MI_DEFAULT and HHO_DEFAULT
(ALL_DEFAULT excluded as the BASELINE identity twin).

In [2]:
# CELL 1 — IMPORTS
import json, numpy as np
from math import sqrt
from scipy.stats import norm, fisher_exact
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import matthews_corrcoef

SEED = 42
B_BOOT = 10000

In [3]:
# CELL 2 — LOAD + ALIGNMENT ASSERTIONS
FS_PATH  = '/kaggle/input/datasets/galibbhai/sealed-predictions/per_patient_sealed_predictions_FS.json'
NB3_PATH = '/kaggle/input/datasets/galibbhai/sealed-predictions/per_patient_sealed_predictions_NB3.json'

fs  = json.load(open(FS_PATH))
nb3 = json.load(open(NB3_PATH))

y = np.asarray(fs['BASELINE']['y_true'])
assert len(y) == 109
for d in fs.values():  assert (np.asarray(d['y_true']) == y).all()
for d in nb3.values(): assert (np.asarray(d['true'])  == y).all()
assert (np.asarray(nb3['GRAPHSAGE']['true']) == y).all()
print(f"Aligned: 109 patients | positives={int(y.sum())} negatives={int((y==0).sum())}")
print("FS configs :", list(fs.keys()))
print("NB3 models :", list(nb3.keys()))

Aligned: 109 patients | positives=36 negatives=73
FS configs : ['PSO_DEFAULT', 'PSO_TUNED', 'HHO_DEFAULT', 'HHO_TUNED', 'MI_DEFAULT', 'MI_TUNED', 'ALL_DEFAULT', 'ALL_TUNED', 'BASELINE']
NB3 models : ['GCN', 'GAT', 'GRAPHSAGE', 'MPNN', 'GIN']


In [4]:
# CELL 3 — TEST FUNCTIONS
def hard(p): return (np.asarray(p) >= 0.5).astype(int)

def _midrank(x):
    J = np.argsort(x); Z = x[J]; N = len(x); T = np.zeros(N); i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]: j += 1
        T[i:j] = 0.5*(i+j-1)+1; i = j
    T2 = np.empty(N); T2[J] = T; return T2

def delong(y_true, pa, pb):
    order = (-y_true).argsort(); m = int(y_true.sum())
    P = np.vstack((pa, pb))[:, order]; n = P.shape[1]-m
    tx = np.empty((2,m)); ty = np.empty((2,n)); tz = np.empty((2,m+n))
    for r in range(2):
        tx[r] = _midrank(P[r,:m]); ty[r] = _midrank(P[r,m:]); tz[r] = _midrank(P[r])
    aucs = tz[:,:m].sum(1)/m/n - (m+1.0)/2.0/n
    cov = np.cov((tz[:,:m]-tx)/n)/m + np.cov(1.0-(tz[:,m:]-ty)/m)/n
    var = cov[0,0]+cov[1,1]-2*cov[0,1]
    if var <= 0: return float(aucs[0]), float(aucs[1]), float('nan')
    z = (aucs[0]-aucs[1])/sqrt(var)
    return float(aucs[0]), float(aucs[1]), float(2*norm.sf(abs(z)))

def mcnemar_exact(y, pr, px):
    hr, hx = hard(pr), hard(px)
    a = int(((hr==y)&(hx==y)).sum()); b = int(((hr==y)&(hx!=y)).sum())
    c = int(((hr!=y)&(hx==y)).sum()); d = int(((hr!=y)&(hx!=y)).sum())
    return b, c, float(mcnemar(np.array([[a,b],[c,d]]), exact=True).pvalue)

def mcc(y, p): return matthews_corrcoef(y, hard(p))

def boot_mcc_delta(y, pr, px, B=B_BOOT, seed=SEED):
    rng = np.random.default_rng(seed); n = len(y)
    hr, hx = hard(pr), hard(px); out = np.empty(B)
    for i in range(B):
        idx = rng.integers(0, n, n)
        out[i] = matthews_corrcoef(y[idx], hr[idx]) - matthews_corrcoef(y[idx], hx[idx])
    return float(np.percentile(out,2.5)), float(np.percentile(out,97.5))

def holm(pairs):
    s = sorted(pairs, key=lambda t: t[1]); m = len(s); adj = {}; run = 0.0
    for i,(nm,p) in enumerate(s):
        run = max(run, (m-i)*p); adj[nm] = min(run, 1.0)
    return adj

def run_family(ref_name, ref_probs, others):
    rows = []
    for nm, px in others:
        b, c, mp = mcnemar_exact(y, ref_probs, px)
        ar, ax, dp = delong(y, ref_probs, px)
        dm = mcc(y, ref_probs) - mcc(y, px)
        lo, hi = boot_mcc_delta(y, ref_probs, px)
        rows.append(dict(cmp=f"{ref_name} vs {nm}", b=b, c=c, mcnemar_p=mp,
                         auc_ref=ar, auc_x=ax, delong_p=dp,
                         mcc_delta=dm, mcc_ci=[lo,hi]))
    hm = holm([(r['cmp'], r['mcnemar_p']) for r in rows])
    hd = holm([(r['cmp'], r['delong_p'])  for r in rows])
    for r in rows:
        r['mcnemar_p_holm'] = hm[r['cmp']]; r['delong_p_holm'] = hd[r['cmp']]
    return rows

def show(rows):
    print(f"{'comparison':24s} {'b/c':>7s} {'McN p':>8s} {'McN Holm':>9s} "
          f"{'DeLong p':>9s} {'DL Holm':>8s} {'dMCC':>8s} {'MCC 95% CI':>19s}")
    for r in rows:
        print(f"{r['cmp']:24s} {str(r['b'])+'/'+str(r['c']):>7s} "
              f"{r['mcnemar_p']:8.4f} {r['mcnemar_p_holm']:9.4f} "
              f"{r['delong_p']:9.4f} {r['delong_p_holm']:8.4f} "
              f"{r['mcc_delta']:+8.4f} [{r['mcc_ci'][0]:+.4f},{r['mcc_ci'][1]:+.4f}]")

In [5]:
# CELL 4 — FAMILY 1: architecture benchmark (reference = GraphSAGE)
fam1 = run_family('GRAPHSAGE', np.asarray(nb3['GRAPHSAGE']['probs']),
                  [(k, np.asarray(nb3[k]['probs'])) for k in ['GCN','GAT','MPNN','GIN']])
print("FAMILY 1 — GraphSAGE vs other architectures (sealed test n=109)\n")
show(fam1)

FAMILY 1 — GraphSAGE vs other architectures (sealed test n=109)

comparison                   b/c    McN p  McN Holm  DeLong p  DL Holm     dMCC          MCC 95% CI
GRAPHSAGE vs GCN            10/3   0.0923    0.1846    0.0243   0.0730  +0.1469 [+0.0000,+0.3004]
GRAPHSAGE vs GAT            11/5   0.2101    0.2101    0.2831   0.5662  +0.1197 [-0.0413,+0.2862]
GRAPHSAGE vs MPNN           11/2   0.0225    0.0674    0.3703   0.5662  +0.1928 [+0.0495,+0.3464]
GRAPHSAGE vs GIN            19/4   0.0026    0.0104    0.0176   0.0705  +0.2439 [+0.0830,+0.4094]


In [6]:
# CELL 5 — FAMILY 2: feature selection vs BASELINE (ALL_DEFAULT excluded)
fam2 = run_family('BASELINE', np.asarray(fs['BASELINE']['probs']),
                  [(k, np.asarray(fs[k]['probs'])) for k in ['MI_DEFAULT','HHO_DEFAULT']])
print("FAMILY 2 — BASELINE vs feature-selected configs (sealed test n=109)\n")
show(fam2)

FAMILY 2 — BASELINE vs feature-selected configs (sealed test n=109)

comparison                   b/c    McN p  McN Holm  DeLong p  DL Holm     dMCC          MCC 95% CI
BASELINE vs MI_DEFAULT       4/5   1.0000    1.0000    0.8113   1.0000  -0.0231 [-0.1478,+0.0949]
BASELINE vs HHO_DEFAULT      5/5   1.0000    1.0000    0.5886   1.0000  -0.0039 [-0.1326,+0.1268]


In [7]:
# CELL 6 — FISHER'S EXACT: single-model discrimination
fisher = {}
for k in ['GCN','GAT','GRAPHSAGE','MPNN','GIN']:
    p = hard(nb3[k]['probs'])
    tn=int(((y==0)&(p==0)).sum()); fp=int(((y==0)&(p==1)).sum())
    fn=int(((y==1)&(p==0)).sum()); tp=int(((y==1)&(p==1)).sum())
    _, pf = fisher_exact([[tp,fp],[fn,tn]])
    fisher[k] = dict(cm=[tn,fp,fn,tp], fisher_p=float(pf))
    print(f"{k:10s} cm(tn{tn},fp{fp},fn{fn},tp{tp})  Fisher p={pf:.3e}")

GCN        cm(tn66,fp7,fn9,tp27)  Fisher p=8.049e-12
GAT        cm(tn65,fp8,fn7,tp29)  Fisher p=6.739e-13
GRAPHSAGE  cm(tn70,fp3,fn6,tp30)  Fisher p=1.385e-17
MPNN       cm(tn66,fp7,fn11,tp25)  Fisher p=2.708e-10
GIN        cm(tn54,fp19,fn5,tp31)  Fisher p=2.291e-09


In [8]:
# CELL 7 — SAVE RESULTS
out = {'n': 109, 'positives': int(y.sum()),
       'family1_architecture': fam1, 'family2_feature_selection': fam2,
       'fisher_single_model': fisher}
with open('statistical_tests_results.json','w') as f:
    json.dump(out, f, indent=2)
print('Saved -> statistical_tests_results.json')

Saved -> statistical_tests_results.json


## Reading guide

- **Holm-adjusted** p-values are the ones to quote; raw p-values inflate the family-wise error rate.
- **McNemar** tests hard-label error; **DeLong** tests AUROC; the **bootstrap CI** is on the MCC delta. A claim is strong only when the direction agrees across tests and the MCC CI excludes zero.
- On n=109 the discordant counts are small, so large MCC deltas can still be non-significant — report effect size and CI alongside the p-value, never the p-value alone.
- Fisher's here tests only whether a single model discriminates above chance; it is not a model-vs-model comparison and every strong model trivially passes.